In [0]:
# -----------------------------------------
# Healthcare Readmission Analytics Project
# Silver Layer Transformation
# -----------------------------------------

from pyspark.sql.functions import *
from pyspark.sql.types import *

# Bronze source table
bronze_table = "workspace.bronze.diabetic_data_bronze"

# Silver target table
silver_table = "workspace.silver.diabetic_data_silver"

In [0]:
# Read Bronze Layer

df_bronze = spark.table(bronze_table)

display(df_bronze)

In [0]:
# Print schema

df_bronze.printSchema()

In [0]:
# Count total records

print(f"Total records: {df_bronze.count()}")

In [0]:
# Count NULL values by column

null_counts = df_bronze.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    
    for c in df_bronze.columns
])

display(null_counts)

In [0]:
# Find columns containing '?'

# Get string-type columns only
string_columns = [field.name for field in df_bronze.schema.fields if field.dataType.simpleString() == 'string']

for column in string_columns:
    
    question_count = (
        df_bronze
        .filter(col(column) == "?")
        .count()
    )

    if question_count > 0:
        print(f"{column}: {question_count}")

In [0]:
# Replace '?' values with NULL

df_cleaned = df_bronze.replace("?", None)

In [0]:
# Validate cleaned data

display(df_cleaned)

In [0]:
# Remove duplicate records

df_cleaned = df_cleaned.dropDuplicates()


In [0]:
# Count records after cleaning

print(f"Records after cleaning: {df_cleaned.count()}")

In [0]:
# Rename important columns

df_cleaned = (
    df_cleaned
    .withColumnRenamed("encounter_id", "patient_encounter_id")
    .withColumnRenamed("patient_nbr", "patient_id")
    .withColumnRenamed("time_in_hospital", "days_in_hospital")
)

In [0]:
# Preview transformed dataframe

display(df_cleaned)

In [0]:
# Create readmission flag

df_cleaned = (
    df_cleaned.withColumn(
        "readmitted_flag",
        
        when(
            col("readmitted") != "NO",
            1
        ).otherwise(0)
    )
)

In [0]:
# Categorize hospital stay duration

df_cleaned = (
    df_cleaned.withColumn(
        "hospital_stay_category",
        
        when(col("days_in_hospital") <= 3, "Short Stay")
        .when(col("days_in_hospital") <= 7, "Medium Stay")
        .otherwise("Long Stay")
    )
)

In [0]:
# Categorize medication usage

df_cleaned = (
    df_cleaned.withColumn(
        "medication_level",
        
        when(col("num_medications") <= 10, "Low")
        .when(col("num_medications") <= 20, "Moderate")
        .otherwise("High")
    )
)

In [0]:
# Add Silver processing timestamp

df_silver = (
    df_cleaned.withColumn(
        "silver_processed_timestamp",
        current_timestamp()
    )
)

In [0]:
# Preview Silver dataframe

display(df_silver)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

In [0]:
# Save Silver dataframe as Delta Table

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
# Read Silver table

df_silver_check = spark.table(silver_table)

display(df_silver_check)

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.silver.diabetic_data_silver;

In [0]:
%sql
SELECT
    readmitted,
    readmitted_flag,
    hospital_stay_category,
    medication_level
FROM workspace.silver.diabetic_data_silver
LIMIT 20;